In [4]:
print(123)

123


In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from openai import OpenAI
client = OpenAI()

In [7]:
def llm(prompt: str) -> str:
    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
    )
    return response.output_text

In [8]:
llm('hey, what\'s up?')

'Hey! Not much on my end—just here to help you out. What’s up with you?'

In [9]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [10]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

It depends on the specific course's enrollment policies and deadlines. Many courses allow late enrollment, while others may have strict cutoff dates. Check the course website or contact the instructor or administration for guidance on joining now.


In [11]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [12]:
answer = llm(prompt)
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [13]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [14]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [15]:
print(courses_raw)

[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 83}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}]


In [16]:
documents = []
url_prefix = "https://datatalks.club/faq/"
for course in courses_raw:
    course_path = course['path']
    course_url = url_prefix + course_path
    response = requests.get(course_url)
    response.raise_for_status()
    course_data = response.json()
    documents.extend(course_data)

len(documents)

1346

In [17]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [18]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [19]:
index.search("I just discovered the course. Can I still join?")

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [20]:
def search(question, course="llm-zoomcamp", num_results=5):

    filter_dict = {"course": course}
    boost_dict = {"question": 2.0, "section": 0.5}
    
    return index.search(
        question, 
        filter_dict=filter_dict, 
        boost_dict=boost_dict,
        num_results=num_results
    )

In [21]:
search_results = search(question)


In [22]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [23]:
USER_PROMPT_TEMPLATE = f"""
Question:
{question}

Context:
{context}
"""

In [24]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [35]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [36]:
prompt = build_prompt(question, search_results)

In [27]:
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.


In [ ]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=prompt
)

In [33]:
response.output_text

'Yes — you can still join.\n\nYou don’t need a confirmation email or any extra approval. You’re accepted, and you can start learning and submitting homework/project work as long as the submission form is still open.\n\nIf you want a certificate, make sure to submit your project before submissions close.'

In [30]:
response.output

[ResponseOutputMessage(id='msg_0eb2f7060ab09714006a311cfb0c608192a6b542b9a4d42570', content=[ResponseOutputText(annotations=[], text='Yes, you can join the course at any time. Just remember that if you want to receive a certificate, you’ll need to submit your project while submissions are still accepted. You can start learning and submitting homework even without formal registration. Enjoy your learning experience!', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]

In [37]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0004335

In [38]:
def llm(instructions, user_prompt, model="gpt-4o-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [39]:
def rag(query, model="gpt-4o-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [40]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted.


In [41]:
rag("How do I get a certificate?")

'Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [42]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()
rag_helper = RAGBase(index=index, course="llm-zoomcamp", llm_client=openai_client)

answer = rag_helper.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while the submissions are still being accepted.


In [43]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [44]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join the course after it has started. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start learning and submitting homework without needing to register, as registration is just a way to gauge interest.'